# Programming Assignment: Improving a RAG System

---

You've made it to the final assignment of the RAG course—congrats on reaching this stage! Now that you’ve got a handle on building RAG systems, it's time to shift gears and make them even better. In this assignment, you’ll dive into:

1. **Cost Measurement**: Figure out the potential costs of running a RAG application.
2. **Prompt Improvement**: Enhance your prompts to speed up response times, while finding the right balance between time, performance, and cost.
3. **Logging System**: Set up a system to keep track of the inputs and outputs during interactions with the RAG system.

These tasks are in your hands now. Good luck, and have fun with it!

---
<h4 style="color:white; font-weight:bold;">USING THE TABLE OF CONTENTS</h4>

JupyterLab provides an easy way for you to navigate through your assignment. It's located under the Table of Contents tab, found in the left panel, as shown in the picture below.

![TOC Location](images/toc.png)

---

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

- All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

- You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

- Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

- - To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: blue; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---


# Table of Contents
- [ 1 - Introduction: Your Role at Fashion Forward Hub](#1)
  - [ 1.1 Importing the libraries](#1-1)
  - [ 1.2 Loading the Weaviate client](#1-2)
- [ 2 - A Quick Recap on the Database Structure](#2)
  - [ 2.1 Products Database](#2-1)
  - [ 2.2 FAQ Database](#2-2)
- [ 3 - Recap on LLM calls and new output](#3)
  - [ 3.1 Function to generate the parameters dictionary](#3-1)
- [ 4 - Improving task handling](#4)
  - [ 4.1 Refactoring the function to decide whether it is an FAQ or product-related question](#4-1)
    - [ Exercise 1](#ex01)
  - [ 4.2 Answering a FAQ question](#4-2)
  - [ 4.3 Querying on FAQ](#4-3)
    - [ Exercise 2](#ex02)
  - [ 4.4 Improving the Decision Between Creative or Technical Product Queries](#4-4)
    - [ Exercise 3](#ex03)
  - [ 4.5 Retrieving the parameters for a given task](#4-5)
- [ 5 - Retrieving Items Based on Metadata from a Query](#5)
  - [ 5.1 Generate metadata](#5-1)
  - [ 5.2 Loading the Weaviate Product Collection](#5-2)
  - [ 5.3 Filtering by Metadata](#5-3)
    - [ Exercise 4](#ex04)
  - [ 5.4 Generating the retrieved items as context](#5-4)
  - [ 5.5 Query on Products](#5-5)
- [ 6 - The final function! ](#6)
  - [ 6.1 The function to rule them all](#6-1)
  - [ 6.2 Logging Time!](#6-2)
    - [ Exercise 5](#ex05)
  - [ 6.3 The ChatBot](#6-3)


<a id='1'></a>
## 1 - Introduction: Your Role at Fashion Forward Hub
---

Congratulations on the success of your ChatBot at Fashion Forward Hub! Customers are thrilled with its functionality, which has significantly reduced calls to customer service and increased sales by providing detailed information and personalized fashion suggestions. However, success has brought some challenges that need your attention:

1. **Rising Costs**: As more customers use the ChatBot, operating costs have increased significantly. This issue has caught the attention of directors, who need better cost monitoring. Currently, there's no system in place to identify where these costs are originating.

2. **High Response Times**: Occasionally, response times are too long for certain user queries, leading to customer dissatisfaction.

As the RAG expert responsible for this ChatBot, your next task is to address these challenges by:

1. Monitoring and controlling costs and response times effectively.
2. Enhancing prompts to strike a balance between cost, performance, and speed.

To accomplish these goals, you will need to develop new functions and refine existing ones in the RAG framework.

<a id='1-1'></a>
### 1.1 Importing the libraries



In [ ]:
import json
from weaviate.classes.query import Filter
import weaviate
import joblib
import pandas as pd

In [ ]:
import flask_app
import weaviate_server
import unittests
from utils import (
    ChatWidget, 
    generate_with_single_input,
    parse_json_output,
    get_filter_by_metadata,
    generate_filters_from_query,
    process_and_print_query,
    print_properties
)

<a id='1-2'></a>
### 1.2 Loading the Weaviate client

In this assignment you will use again the Weaviate API to load the vector database. Do not worry, you won't need to load the database. It is already given to you!

In [ ]:
client = weaviate.connect_to_local(port=8079, grpc_port=50050)

<a id='2'></a>
## 2 - A Quick Recap on the Database Structure
---

Let's have a quick recap on both the FAQ and Products databases.

Recall:
- Product database: Contains the products and their information.
- FAQ database: Contains the FAQ data.

<a id='2-1'></a>
### 2.1 Products Database

Let's explore the products database that Fashion Forward Hub has available. To make it easier to understand, let's load it as a list of JSON files first.

In [ ]:
# Loading products data
products_data = joblib.load('dataset/clothes_json.joblib')

In [ ]:
# Let's get one example
products_data[0]

The features each product has are:

- **Gender:** Target audience for the product, such as "Men," "Women," or "Unisex."
- **Master Category:** Broad classification like "Apparel" or "Footwear."
- **Sub Category:** Specific category within a master category, such as "Topwear."
- **Article Type:** Exact type of product, e.g., "Shirts" or "Jackets."
- **Base Colour:** Main color of the product, important for customer choice.
- **Season:** Intended season for the product, e.g., "Summer" or "Winter."
- **Year:** Year of release or collection.
- **Usage:** Intended use or occasion, like "Casual" or "Formal."
- **Product Display Name:** Descriptive name used in marketing.
- **Price:** Cost of the product.
- **Product ID:** Unique identifier for managing and tracking inventory.

<a id='2-2'></a>
### 2.2 FAQ Database

Now, let's load the FAQ database and explore it.

In [ ]:
faq = joblib.load("dataset/faq.joblib")

In [ ]:
# Get an example
faq[:2]

The FAQs are organized in a list, where each entry is a dictionary containing the following keys: `question`, `answer`, and `type`.

<a id='3'></a>
## 3 - Recap on LLM calls and new output
---

Let's recap the previous function you used to generate prompts. Now it has been enhanced with a new output parameter: `total_tokens`, which will be used to compute the costs.

Recall:

```Python
generate_with_single_input(prompt: str,
                           role: str = 'user',
                           top_p: float = 1,
                           temperature: float = 1,
                           max_tokens: int = 500,
                           model: str = "meta-llama/Llama-3.2-3B-Instruct-Turbo")
```

Let's now understand the new behavior:

In [ ]:
# The output is a dictionary containing the role and content from the LLM call, as well as the token usage.:
result = generate_with_single_input("What are the primary colors?")
print(result['content'])

In [ ]:
# The total tokens count (input + output) for this is:
print(result['total_tokens'])

Note that there is now a new key called `total_tokens`. Usually, LLM costs are measured by token count (both input and output).

<a id='3-1'></a>
### 3.1 Function to generate the parameters dictionary

This function was used in the previous assignment. You will need it in this assignment too; therefore, it will be provided to you.

In [ ]:
def generate_params_dict(
    prompt: str,
    temperature: float = 1.0,
    role: str = 'user',
    top_p: float = 1.0,
    max_tokens: int = 500,
    model: str = "meta-llama/Llama-3.2-3B-Instruct-Turbo"
) -> dict:
    """
    Generates a dictionary of parameters for calling a Language Learning Model (LLM),
    allowing for the customization of several key options that can affect the output from the model. 

    Args:
        prompt (str): The input text that will be provided to the model to guide text generation.
        temperature (float): A value between 0 and 1 that controls the randomness of the model's output; 
            lower values result in more repetitive and deterministic results, while higher values enhance randomness.
        role (str): The role designation to be used in context, typically identifying the initiator of the interaction.
        top_p (float): A value between 0 and 1 that manages diversity through the technique of nucleus sampling; 
            this parameter limits the set of considered words to the smallest possible while maintaining 'top_p' cumulative probability.
        max_tokens (int): The maximum number of tokens that the model is allowed to generate in response, where a token can 
            be as short as one character or as long as one word.
        model (str): The specific model identifier to be utilized for processing the request. This typically specifies both 
            the version and configuration of the LLM to be employed.

    Returns:
        dict: A dictionary containing all specified parameters which can then be used to configure and execute a call to the LLM.
    """
    # Create the dictionary with the necessary parameters
    kwargs = {
        "prompt": prompt,
        "role": role,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "model": model
    }
    return kwargs

In [ ]:
kwargs = generate_params_dict("Solve 3x^2 + 5 = 0")
print(kwargs)

In [ ]:
# Now you can call the LLM 
result = generate_with_single_input(**kwargs)
print(f"Content: {result['content']}\n\nTotal Tokens: {result['total_tokens']}")

<a id='4'></a>
## 4 - Improving task handling
---

<a id='4-1'></a>
### 4.1 Refactoring the function to decide whether it is an FAQ or product-related question


This is the previous function you built to check if a query is related to FAQs or Products. Now, there are two main changes:

1. It now returns the total number of tokens used in the process (including both the input and output of the LLM) — this value is already given to you.
2. It has a new parameter called `simplified`. If `True`, it uses a shorter prompt — that’s your task in this exercise.

<a id='ex01'></a>

<a id='ex01'></a>
### Exercise 1

---

In this exercise, you need to improve the prompt that checks whether a query is FAQ-related or Product-related. The new prompt must use less than 130 tokens in total. It also needs to keep the same classification accuracy for the test set below — that means it must give the same result for each query.

<details>
    <summary style="color: green;"><strong>Hint 1</strong></summary>
Try making the text shorter and removing some examples. Make sure to include the user query in the prompt.
</details>

<details>
    <summary style="color: green;"><strong>Hint 2</strong></summary>
If your prompt doesn’t work for a specific query, consider adding that query (or a more difficult version of it) as an example. The better you handle tricky cases, the more reliable your prompt will be.
</details>


In [ ]:
# GRADED CELL 

def check_if_faq_or_product(query, simplified = False):
    """
    Determines whether a given instruction prompt is related to a frequently asked question (FAQ) or a product inquiry.

    Parameters:
    - query (str): The instruction or query that needs to be labeled as either FAQ or Product related.
    - simplified (bool): If True, uses a simplified prompt.

    Returns:
    - str: The label 'FAQ' if the prompt is deemed a frequently asked question, 'Product' if it is related to product information, or
      None if the label is inconclusive.
    """
 
    
    # If not simplified, uses a more complex prompt
    if not simplified:
        PROMPT = f"""Label the following instruction as an FAQ related answer or a product related answer.
        Product related answers are answers specific about product information or that needs to use the products to give an answer. Products are clothes from a store.
        Examples:
                Is there a refund for incorrectly bought clothes? Label: FAQ
                Tell me about the cheapest T-shirts that you have. Label: Product
                Do you have blue T-shirts under 100 dollars? Label: Product
                What are the available sizes for the t-shirts? Label: FAQ
                How can I find the promotions? Label: FAQ
                Give me ideas for a sunny look. Label: Product
        Return only one of the two labels: FAQ or Product.
        Instruction: {query}
                 """
    ### START CODE HERE ###

    # If simlpified, uses a simplified prompt.
    else:
        PROMPT = None
    ### END CODE HERE ###
        

    # Get the kwargs dictinary to call the llm, with PROMPT as prompt, low temperature (0 or near 0) and max_tokens = 1
    kwargs = generate_params_dict(PROMPT, temperature = 0, max_tokens = 1)

    # Call generate_with_single_input with **kwargs
    response = generate_with_single_input(**kwargs) 
    
    # Get the Label by accessing the content key of the response dictionary
    label = response['content'] 
    total_tokens = response['total_tokens']

    return label, total_tokens

In [ ]:
unittests.test_check_if_faq_or_product(check_if_faq_or_product)

Let's test both versions:

In [ ]:
queries = [
    'What is your return policy?', 
    'Give me three examples of blue Tshirts you have available.', 
    'How can I contact the user support?', 
    'Do you have blue Dresses?',
    'Create a look suitable for a wedding party happening during dawn.'
]

labels = ['FAQ', 'Product', 'FAQ', 'Product', 'Product']

for query, correct_label in zip(queries, labels):
    # Call check_if_faq_or_product and store the results
    response_std, tokens_std = check_if_faq_or_product(query, simplified=False)
    response_simp, tokens_simp = check_if_faq_or_product(query, simplified=True)
    
    # Print results
    process_and_print_query(query, correct_label, response_std, tokens_std, response_simp, tokens_simp)

<a id='4-2'></a>
### 4.2 Answering a FAQ question

Let's recap how to generate the FAQ layout. This won't be touched in this assignment. This function, given a list of dictionaries with the FAQ questions, returns a formatted string with the question/answer pairs.

In [ ]:
def generate_faq_layout(faq_dict):
    """
    Generates a formatted string layout for a list of FAQs.

    This function iterates through a dictionary of frequently asked questions (FAQs) and constructs
    a string where each question is followed by its corresponding answer and type.

    Parameters:
    - faq_dict (list): A list of dictionaries, each containing keys 'question', 'answer', and 'type' 
      representing an FAQ entry.

    Returns:
    - str: A string representing the formatted layout of FAQs, with each entry on a separate line.
    """
    # Initialize an empty string
    t = ""
    
    # Iterate over every FAQ question in the FAQ list
    for f in faq_dict:
        # Append the question with formatted string (remember to use f-string and access the values as f['question'], f['answer'] and so on)
        # Also, do not forget to add a new line character (\n) at the end of each line.
        t += f"Question: {f['question']} Answer: {f['answer']} Type: {f['type']}\n" 

    return t

In [ ]:
# You can generate a full faq_layout with the entire FAQ questions
faq_layout = generate_faq_layout(faq)
print(faq_layout[:1000])

In [ ]:
# You can choose some faq questions and generate a layout from them. 
# They just need to be in a list with dictionaries with the necessary keys: 'question', 'answer' and 'type'
print(generate_faq_layout(faq[1:2]))

<a id='4-3'></a>
### 4.3 Querying on FAQ

In the previous assignment, the entire FAQ was added in the query. This approach is useful to provide the entire information to the LLM, but it significantly increases token usage and execution time. Now that you are refining your chatbot, it's time to use a more efficient collection to handle it! Let's load the collection.

In [ ]:
faq_collection = client.collections.get("Faq")

Let's add the FAQ questions into a collection.

In [ ]:
from tqdm import tqdm
from weaviate.util import generate_uuid5
# Set up a batch process with specified fixed size and concurrency
with faq_collection.batch.fixed_size(batch_size=5, concurrent_requests=1) as batch:
    # Iterate over a subset of the dataset
    for document in tqdm(faq):
        # Generate a UUID based on the chunk text for unique identification
        uuid = generate_uuid5(document['question'])

        # Add the chunk object to the batch with properties and UUID
        batch.add_object(
            properties=document,
            uuid=uuid,
        )

Now you can query them! Let's go over an example.

In [ ]:
res = faq_collection.query.near_text("What is the return policy?", limit = 5)

In [ ]:
for obj in res.objects:
    print_properties(obj)

<a id='ex02'></a>

<a id='ex02'></a>
### Exercise 2
---

Below is the function used to answer a FAQ question.
**It will only run if the question is already labeled as a FAQ.**
You’ve seen this question in a previous assignment, but now there’s one change:

1. A new parameter called `simplified` was added.
   This controls whether the function uses the full `faq` list or a smaller selection from it.
   If `simplified` is `True`, you should run a semantic search on the FAQ collection and use only the top 5 results.

That’s your task in this exercise.

Your solution must return **fewer than 500 tokens** for the query below.

<details>
    <summary style="color: green;"><strong>Hint 1</strong></summary>
To run a semantic search on the Weaviate database (Module 3), use:  
<code>faq_collection.query.near_text(query, limit=5)</code>
</details>

<details>
    <summary style="color: green;"><strong>Hint 2</strong></summary>
    Don’t forget to call <code>generate_faq_layout()</code> with the result list as its argument.
</details>

In [ ]:
# GRADED CELL 

def query_on_faq(query, simplified = False, **kwargs):
    """
    Constructs a prompt to query an FAQ system and generates a response.

    This function integrates an FAQ layout into the prompt to help generate a suitable answer to the given query
    using a language model. It supports additional keyword arguments to customize the prompt generation process.

    Parameters:
    - query (str): The query about which the function seeks to provide an answer from the FAQ.
    - simplified (bool): If True, uses semantic search to extract a relevant subset of FAQ questrions
    - **kwargs: Optional keyword arguments for extra configuration of prompt parameters.

    Returns:
    - str: The response generated from the language model based on the input query and FAQ layout.

    """

    
    # If not simplified, generate the faq layout with the entire FAQ questions
    if not simplified:
        faq_layout = generate_faq_layout(faq)
        
        # Generate the prompt
        PROMPT = f"""You will be provided with an FAQ for a cloth store. 
    Answer the instruction based on it. You might use more than one question and answer to make your answer. Only answer the question and do not mention that you have access to a FAQ. 
    <scratchpad>
    PROVIDED FAQ: {faq_layout}
    </scratchpad>
    Question: {query}
        """ 
        
    ### START CODE HERE ###
    
    else:
        # Get the 5 most relevant FAQ objects, in this case limit = None
        results = None
        # Transform the results in a list of dictionary
        results = [x.properties for x in results.objects] 
        # Reverse the order to add the most relevant objects in the bottom, so it gets closer to the end of the input
        results.reverse() 
        # Generate the faq layout with the new list of FAQ questions `results`
        faq_layout = None
        
    ### END CODE HERE ###
        
        # Different prompt to deal with this new scenario. 
        PROMPT = (f"You will be provided with a query for a cloth store regarding FAQ. It will be provided relevant FAQ from the cloth store." 
    f"Answer the query based on the relevant FAQ provided. They are ordered in decreasing relevance, so the first is the most relevant FAQ and the last is the least relevant."  
    f"Answer the instruction based on them. You might use more than one question and answer to make your answer. Only answer the question and do not mention that you have access to a FAQ.\n"  
    f"<scratchpad>\n"  
    f"RELEVANT FAQ ITEMS:\n{faq_layout}\n"  
    f"</scratchpad>\n" 
    f"Query: {query}")


    
    # Generate the parameters dict with PROMPT and **kwargs 
    kwargs = generate_params_dict(PROMPT, **kwargs) 
    
    return kwargs

In [ ]:
unittests.test_query_on_faq(query_on_faq)

In [ ]:
# Get the dictionary of arguments
kwargs = query_on_faq("I got my cloth but I didn't like it. How can I return it?")

In [ ]:
# The number of split tokens in this prompt is:
print(len(kwargs['prompt'].split()))

Note: The number mentioned above doesn’t match the exact token count the model will use. Tokenization is more complex than just splitting by words, so the actual count might be higher. Still, this gives you a rough idea of the text size and is useful for comparison.

In [ ]:
# Run the inference
content = generate_with_single_input(**kwargs)

Let's check the content without the simplified version:

In [ ]:
print(content['content'])

In [ ]:
# Get the total tokens
print(content['total_tokens'])

Note that for one query, the total tokens is around 1220.

Now let's check the simplified version.

In [ ]:
# Get the dictionary of arguments
kwargs = query_on_faq("I got my cloth but I didn't like it. How can I return it?", simplified = True)

In [ ]:
# The number of split tokens in this prompt is:
print(len(kwargs['prompt'].split()))

In [ ]:
# Run the inference
content = generate_with_single_input(**kwargs)

In [ ]:
print(content['content'])

In [ ]:
# Get the total tokens
print(content['total_tokens'])

Note that the answer is still correct and the final token count is way smaller!

<a id='4-4'></a>

<a id='4-4'></a>
### 4.4 Improving the Decision Between Creative or Technical Product Queries

<a id='ex03'></a>

<a id='ex03'></a>
### Exercise 3

---

This task is similar to what you did when deciding whether a query was about a product or a FAQ. The goal is the same: reduce token usage while keeping good accuracy.

This function has two updates compared to the previous version:

1. It now returns the total number of tokens used during processing.
2. It includes a new argument called `simplified`.

Your solution must meet both of the following conditions:

* Accuracy of at least **80%** on the test set (you can get **at most one** question wrong).
* Use **fewer than 150 tokens** for **every** query.

<details>
    <summary style="color: green;"><strong>Hint 1</strong></summary>
Try shortening the examples and removing any that aren’t essential. Don’t forget to include the query in the prompt!
</details>

<details>
    <summary style="color: green;"><strong>Hint 2</strong></summary>
If your prompt struggles with a particular query, try adding it—or a more challenging version of it—as an example. The more difficult edge cases you cover, the better the model will perform.
</details>

In [ ]:
# GRADED CELL 

def decide_task_nature(query, simplified = True):
    """
    Determines the nature of a query, labeling it as either creative or technical.

    This function constructs a prompt for a language model to decide if a given query requires a creative response,
    such as making suggestions or composing ideas, or a technical response, like providing product details or prices.

    Parameters:
    - query (str): The query to be evaluated for its nature.
    - simplified (bool): If True, uses a simplified prompt.

    Returns:
    - str: The label 'creative' if the query requires creative input, or 'technical' if it requires technical information.
    """


    
    if not simplified:
        PROMPT = f"""Decide if the following query is a query that requires creativity (creating, composing, making new things) or technical (information about products, prices etc.). Label it as creative or technical.
          Examples:
          Give me suggestions on a nice look for a nightclub. Label: creative
          What are the blue dresses you have available? Label: technical
          Give me three Tshirts for summer. Label: technical
          Give me a look for attending a wedding party. Label: creative
          Give me suggestions on clothes that match a green Tshirt. Label: creative
          I would like a suggestion on which products match a green Tshirt I already have. Label: creative

          Query to be analyzed: {query}. Only output one token with the label
          """

    # If simplified, uses a simplified query

    ### START CODE HERE ###

    else:
        PROMPT = None
    ### END CODE HERE ###

    # Generate the kwards dictionary by passing the PROMPT, low temperature and max_tokens = 1
    kwargs = generate_params_dict(PROMPT, temperature = 0, max_tokens = 1)

    # Get the response with generate_with_single_input and **kwargs
    response = generate_with_single_input(**kwargs) 

    # Get the label
    label = response['content'] 
    
    total_tokens = response['total_tokens']
    
    
    return label, total_tokens

In [ ]:
unittests.test_decide_task_nature(decide_task_nature)

In [ ]:
queries = ["Give me two sneakers with vibrant colors.",
           "What are the most expensive clothes you have in your catalogue?",
           "I have a green Dress and I like a suggestion on an accessory to match with it.",
           "Give me three trousers with vibrant colors you have in your catalogue.",
           "Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather."
           ]

labels = ['technical', 'technical', 'creative', 'technical', 'creative']

In [ ]:
for query, correct_label in zip(queries, labels):
    response, total_tokens = decide_task_nature(query, simplified = True)
    label = response
    if label == correct_label:
        label = "\033[32m" + label + "\033[0m" 
    else:
        label = "\033[31m" + label + "\033[0m"
    if total_tokens > 150:
        total_tokens = "\033[31m"  + str(total_tokens) + "\033[0m"
    else:
        total_tokens = "\033[32m"  + str(total_tokens) + "\033[0m"
    print(f"Query: {query} Label Predicted: {label}. Correct Label: {correct_label} Total Tokens: {total_tokens}")

<a id='4-5'></a>
### 4.5 Retrieving the parameters for a given task

This is the same function as the previous assignment. It uses different parameters for creative and technical questions.

In [ ]:
def get_params_for_task(task):
    """
    Retrieves specific language model parameters based on the task nature.

    This function provides parameter sets tailored for creative or technical tasks to optimize
    language model behavior. For creative tasks, higher randomness is encouraged, while technical
    tasks are handled with more focus and precision. A default parameter set is provided for unexpected cases.

    Parameters:
    - task (str): The nature of the task ('creative' or 'technical').

    Returns:
    - dict: A dictionary containing 'top_p' and 'temperature' settings for the specified task.
    """
    # Create the parameters dict for technical and creative tasks
    PARAMETERS_DICT = {"creative": {'top_p': 0.9, 'temperature': 1},
                       "technical": {'top_p': 0.7, 'temperature': 0.3}} 
    
    # If task is techincal, return the value for the key technical in PARAMETERS_DICT
    if task == 'technical':
        param_dict = PARAMETERS_DICT['technical'] 

    # If task is creative, return the value for the key creative in PARAMETERS_DICT
    if task == 'creative':
        param_dict = PARAMETERS_DICT['creative'] 

    # If task is a different value, fallback to another set of parameters
    else: # Fallback to a standard value
        param_dict = {'top_p': 0.5, 'temperature': 1} 

    
    return param_dict

<a id='5'></a>
## 5 - Retrieving Items Based on Metadata from a Query

---

In the previous framework, when a query is identified as a product query, you need to find and return relevant products from the vector database. This process works in three main steps:

1. **Generate a metadata JSON** — Use the LLM to guess likely values for some product categories based on the query.
2. **Run a semantic search** — Use those values as filters when querying the database.
3. **Return the results** — Provide the most relevant products found.

The metadata should include values for the following features:

* Gender
* Master Category
* Article Type
* Base Color
* Season
* Usage

These categories offer a good trade-off between being specific enough to improve relevance and general enough to avoid missing results. Using too many or overly detailed filters could lead to no matches, while including too few could make the query too broad and inefficient. This balance helps keep the system fast, accurate, and cost-effective in real-world use.

In [ ]:
# Let's remember the data structure of a product
products_data[0]

This is a dictionary with every possible value for the categories the LLM can pick from to generate a JSON. Note that this dictionary can become huge.

In [ ]:
# Run this cell to generate the dictionary with the possible values for each key
values = {}
for d in products_data:
    for key, val in d.items():
        if key in ('product_id', 'price', 'productDisplayName', 'subCategory', 'year'):
            continue
        if key not in values.keys():
            values[key] = set()
        values[key].add(val)

In [ ]:
# Example of possible values for the feature 'season'
values['season']

<a id='5-1'></a>
### 5.1 Generate metadata

This function generates a metadata JSON with possible values for each cloth category. The possible values are passed through the dictionaty "values". 

Note that the prompt is huge. Let's investigate the total tokens for a query.

In [ ]:
def generate_metadata_from_query(query):
    """
    Generates metadata in JSON format based on a given query to filter clothing items.

    This function constructs a prompt for a language model to create a JSON object that will
    guide the filtering of a vector database query for clothing items. It takes possible values from
    a predefined set and ensures only relevant metadata is included in the output JSON.

    Parameters:
    - query (str): The query describing specific clothing-related needs.

    Returns:
    - str: A JSON string representing metadata with keys like gender, masterCategory, articleType,
      baseColour, price, usage, and season. Each value in the JSON is within a list, with prices specified
      as a dict containing "min" and "max" values. Unrestricted keys should use ["Any"] and unspecified
      prices should default to {"min": 0, "max": "inf"}.
    """

    # Set the prompt. Remember to include the query, the desired JSON format, the possible values (passing {values} at some point) 
    # and explain to the LLM what is going on. 
    # Explicitly tell the llm to include gender, masterCategory, ArticleType, baseColour, price, usage and season as keys.
    # Also mention to the llm that price key must be a json with "min" and "max" values (0 if no lower bound and inf if no upper bound)
    # If there is no price set, add min = 0 and max = inf.
    PROMPT = f"""
    One query will be provided. For the given query, there will be a call on vector database to query relevant cloth items. 
    Generate a JSON with useful metadata to filter the products in the query. Possible values for each feature is in the following json: {values}

    Provide a JSON with the features that best fit in the query (can be more than one, write in a list). Also, if present, add a price key, saying if there is a price range (between values, greater than or smaller than some value).
    Only return the JSON, nothing more. price key must be a json with "min" and "max" values (0 if no lower bound and inf if no upper bound). 
    Always include gender, masterCategory, ArticleType, baseColour, price, usage and season as keys. All values must be within lists.
    If there is no price set, add min = 0 and max = inf.
    Only include values that are given in the json above. 
    
    Example of expected JSON:

    {{
    "gender": ["Women"],
    "masterCategory": ["Apparel"],
    "articleType": ["Dresses"],
    "baseColour": ["Blue"],
    "price": {{"min": 0, "max": "inf"}},
    "usage": ["Formal"],
    "season": ["All seasons"]
    }}

    Query: {query}
             """

    # Generate the response with the generate_with_single_input, PROMPT, temperature = 0 (low randomness) and max_tokens = 1500.
    response = generate_with_single_input(PROMPT, temperature = 0, max_tokens = 1500) # @REPLACE EQUALS None

    # Get the content
    content = response['content']
    
    total_tokens = response['total_tokens']

    
    return content, total_tokens

In [ ]:
content, total_tokens = generate_metadata_from_query("Create a look for a man that suits a sunny day in the park. I don't want to spend more than 300 dollars on each piece.")

In [ ]:
print(content)

In [ ]:
print(total_tokens)

So far, each product query has involved processing around **1,500 tokens**—mainly because we generate a set of filters across multiple categories before searching.

You will now **simplify** this process.

Instead of creating detailed filters for each category (like gender, color, etc.), the system will just use **semantic search directly on the user query**. This means:

* No more generating metadata.
* Just take the user’s question and run a semantic search on the product collection.

This approach is faster, uses fewer tokens, and is still effective for most queries.

<a id='5-2'></a>
### 5.2 Loading the Weaviate Product Collection

Now it is time to work with the Weaviate collection. It is already given to you and it is the product_data you saw before, but added as a Weaviate collection, so we can query with semantic search and metadata filtering.

In [ ]:
products_collection = client.collections.get('products')

In [ ]:
len(products_collection)

<a id='5-3'></a>

<a id='5-3'></a>
### 5.3 Filtering by Metadata

The functions used to filter by metadata have been moved to the **`utils.py`** file.
You can find this file in the **File Browser** on the left panel.

You worked with these functions in the previous assignment, but for this one, **you won’t need to use them directly**.

So, let’s go ahead and jump into the exercise!

<a id='ex04'></a>
### Exercise 4

---


The next function retrieves relevant products based on a query.
It’s a modified version of the one you used previously, with one key change:

* It now includes a boolean parameter called `simplified`.
* If `simplified` is `True`, the function **must skip metadata filtering** and perform a **simple semantic search** using the query.
* Choose an appropriate limit—5 may be too low. In the previous scenario, 20 items were returned, so you might want to stick with that.

Therefore, when `simplified = True`, you should only run a semantic search—**no metadata filters should be applied**.

<details>
  <summary style="color: green;"><strong>Hint 1</strong></summary>
  Use the <code>products_collection</code> (not <code>faq_collection</code>).  
  To query the Weaviate database, use:  
  <code>products_collection.query.near_text(query, limit=pick_your_limit)</code>
</details>

In [ ]:
# GRADED CELL

def get_relevant_products_from_query(query, simplified = False):
    """
    Retrieve the most relevant products for a given query by applying semantic search and optional filters.

    This function generates metadata filters from the query and uses them to search for products 
    that best match the intended criteria. If `simplified` is True, it performs only a basic semantic 
    search with no filters. If the filtered search returns too few results, it progressively reduces 
    filtering constraints based on the predefined importance of each filter.

    Parameters:
    query (str): The query string used to search for relevant products.
    simplified (bool): If True, only a simple semantic search is performed without any metadata filters.

    Returns:
    list: A list of product objects that are most relevant to the query.
    total_tokens: The number of tokens used in the LLM call. Returns 0 if simplified search is used.
    """
    ### START CODE HERE ###
    
    # If simplified, just do a semantic search with 20 objects and return it
    if simplified:
        res = None
    
    ### END CODE HERE ###
    
        return res, 0  # Total tokens in this case is 0 because there was no LLM call!
    

    # If not simplified, perform the previous workflow by generating the filters and then doing a semantic search with them
    filters, total_tokens = generate_filters_from_query(query)  # Generate filters based on the query

    # Check if there are no applicable filters
    if filters is None or len(filters) == 0:
        # Query the collection without filters, using the query text for relevance
        res = products_collection.query.near_text(query, limit=20).objects
        return res, total_tokens

    # Query with filters and limit to the top 20 relevant objects
    res = products_collection.query.near_text(query, filters=Filter.all_of(filters), limit=20).objects

    # If the result set contains fewer than 10 products, try reducing filters to broaden the search
    importance_order = ['baseColour', 'masterCategory', 'usage', 'masterCategory', 'season', 'gender']

    if len(res) < 10:
        # Iterate through the importance order of filters
        for i in range(len(importance_order)):
            # Create a list of filters that excludes less important ones
            filtered_filters = [x for x in filters if x.target not in importance_order[i+1:]]
            
            # Re-query with the reduced set of filters
            res = products_collection.query.near_text(query, filters=Filter.all_of(filtered_filters), limit=20).objects
            
            # If sufficient products have been found, return early
            if len(res) >= 5:
                return res, total_tokens

    return res, total_tokens  # Return the final set of relevant products

In [ ]:
query = "Give me three Tshirts to use in sunny days"

In [ ]:
t, total_tokens = get_relevant_products_from_query(query)

In [ ]:
total_tokens

Around 1500 tokens for this query! Let's try with the simplified version

In [ ]:
t, total_tokens = get_relevant_products_from_query(query, simplified = True)

In [ ]:
total_tokens

Note that this query took 0 tokens, as it didn't use the LLM. It directly used the query to retrieve the objects that are in the vector database.

In [ ]:
# Test your solution!
unittests.test_get_relevant_products_from_query(get_relevant_products_from_query)

<a id='5-4'></a>

<a id='5-4'></a>
### 5.4 Generating the retrieved items as context

Now, for the given retrieved items, let's generate a simple context in the following format:

```
Product Name: Inkfruit Men's Little Bit More T-shirt. Product Category: Apparel. Product Usage: Casual. Product Gender: Men. Product Type: T-shirts. Product Category: Topwear. Product Color: Yellow. Product Season: Summer. Product Year: 2011.
```

In [ ]:
def generate_items_context(results):
    """
    Compile detailed product information from a list of result objects into a formatted string.

    Parameters:
    results (list): A list of result objects, each having a `properties` attribute that is a dictionary 
                    containing product attributes such as 'product_id', 'productDisplayName', 
                    'masterCategory', 'usage', 'gender', 'articleType', 'subCategory', 
                    'baseColour', 'season', and 'year'.

    Returns:
    str: A multi-line string where each line contains the formatted details of a single product.
         Each product detail includes the product ID, name, category, usage, gender, type, color, 
         season, and year.
    """
    t = ""  # Initialize an empty string to accumulate product information

    for item in results:  # Iterate through each item in the results list
        item = item.properties  # Access the properties dictionary of the current item

        # Append formatted product details to the output string
        t += (
            f"Product ID: {item['product_id']}. "
            f"Product name: {item['productDisplayName']}. "
            f"Product Category: {item['masterCategory']}. "
            f"Product usage: {item['usage']}. "
            f"Product gender: {item['gender']}. "
            f"Product Type: {item['articleType']}. "
            f"Product Category: {item['subCategory']} "
            f"Product Color: {item['baseColour']}. "
            f"Product Season: {item['season']}. "
            f"Product Year: {item['year']}.\n"
        )

    return t  # Return the complete formatted string with product details

In [ ]:
print(generate_items_context(t)[:1000])

<a id='5-5'></a>
### 5.5 Query on Products

The next function will answer a product query. 

In [ ]:
def query_on_products(query, simplified = False):
    """
    Execute a product query process to generate a response based on the nature of the query.

    Parameters:
    query (str): The input query string that needs to be analyzed and answered using product data.
    task_nature_prompt_function (func): The prompt function to be used to decide the task nature (if creative of technical)
    simplified (bool): If True, does not use LLM to generate metadata for filtering

    Returns:
    dict: A dictionary of keyword arguments (`kwargs`) containing the prompt and additional settings 
          for creating a response, suitable for input to an LLM or other processing system.
    int: Number of tokens used in the process to create the kwargs dictionary

    Outputs:
    str: The content of the generated response from the LLM based on the provided query and product 
         information.
    """
    total_tokens = 0
    # Determine if the query is technical or creative in nature
    
    query_label, tokens = decide_task_nature(query, simplified = simplified)
    
    # Sum the tokens used to decide the task nature (creative or technical)
    total_tokens += tokens

    # Obtain necessary parameters based on the query type
    parameters_dict = get_params_for_task(query_label)
    
    # Retrieve products that are relevant to the query
    relevant_products, tokens = get_relevant_products_from_query(query, simplified = simplified)
    
    # Sum the tokens used to get relevant products 
    total_tokens += tokens
     
    # Create a context string from the relevant products
    context = generate_items_context(relevant_products)

    # Construct a prompt including product details and the query. Remember to add the context and the query in the prompt, also, ask the LLM to provide the product ID in the answer
    PROMPT = (
        f"Given the available set of cloth products, "
        f"answer the question that follows. "
        f"Provide the item ID in your answers. "
        f"The other information might be provided but not necessarily all of them, pick only the relevant ones for the given query, don't be too long on describing the items features.\n"
        f"If no number of products is mentioned in the query, select at most five to show."
        f"CLOTH PRODUCTS AVAILABLE: {context}\n"
        f"QUERY: {query}"
    )
    
    # Generate kwargs (parameters dict) for parameterized input to the LLM with , Prompt, role = 'assistant' and **parameters_dict
    kwargs = generate_params_dict(PROMPT, role='assistant', **parameters_dict) 

    
    return kwargs, total_tokens

Let's check with both the previous setup and the enhanced setup


#### Previous setup with simplified = False

In [ ]:
kwargs, total_tokens = query_on_products('Make a wonderful look for a man attending a wedding party happening during night.', simplified = False)

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['content'])

Now let's sum the total tokens to generate the kwargs dictionary and the total tokens used in the final execution.

In [ ]:
print(f"Total tokens used in the query is: {total_tokens + result['total_tokens']}")

**New setup with <code>simplified = True</code>**

In [ ]:
kwargs, total_tokens = query_on_products('Make a wonderful look for a man attending a wedding party happening during night.', simplified = True)

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['content'])

In [ ]:
print(f"Total tokens used in the query is: {total_tokens + result['total_tokens']}")

And the total tokens used in one query was way lower than before!

<a id='6'></a>
## 6 - The final function! 
---
<a id='6-1'></a>
### 6.1 The function to rule them all

Now let's consolidate the functions

The function will:

1. Check if the query is FAQ or Product
2. If FAQ, runs the FAQ related workflow
3. If Product, runs the Product related workflow
4. Add the information into a dataframe

It returns the kwargs dict with the appropriate arguments and the total tokens used to get to the kwargs dict.

In [ ]:
def answer_query(query, simplified=False):
    """
    Processes a user's query to determine its type (FAQ or Product) and executes the appropriate workflow.
    
    Parameters:
    - query (str): The query string provided by the user.
    - simplified (bool): If True, uses a simplified version of the method. Defaults to False.
    
    Returns:
    - dict: A dictionary containing keyword arguments for further processing.
      If the query is neither FAQ nor Product-related, returns a default response dictionary instructing
      the assistant to answer based on existing context.
    """
    # Initialize the total tokens used to zero
    total_tokens = 0
    
    # Determine if the query is FAQ or Product and get the token count for this step
    label, tokens = check_if_faq_or_product(query, simplified=simplified)
    
    # Sum the tokens
    total_tokens += tokens
    
    # If the query is neither FAQ nor Product, return a default response
    if label not in ['FAQ', 'Product']:
        return {
            "role": "assistant",
            "prompt": (f"User provided a question that does not fit FAQ or Product-related categories. "
                       f"Answer it based on the context you already have. Query provided by the user: {query}")
        }
    
    # Process the query based on its label
    if label == 'FAQ':
        # Handle FAQ-related queries
        kwargs = query_on_faq(query, simplified=simplified)
    elif label == 'Product':
        try:
            # Handle Product-related queries, with error handling in place
            kwargs, tokens = query_on_products(query, simplified=simplified)
            # Add the tokens to the total tokens
            total_tokens += tokens
        except Exception:
            # Return an error response if an exception occurs during querying
            return {
                "role": "assistant",
                "prompt": (f"User provided a question that broke the querying system. "
                           f"Instruct them to rephrase it. Answer it based on the context you already have. "
                           f"Query provided by the user: {query}")
            }, total_tokens
    
    # Return the kwargs and total_tokens for further processing
    return kwargs, total_tokens

In [ ]:
kwargs, total_tokens = answer_query("Give me three examples of blue t-shirts available on your catalogue.", simplified = False)

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['content'])

In [ ]:
# To get the total tokens for the call, we must sum the total_tokens to get the kwargs dictionary + total tokens from the LLM call
total_tokens + result['total_tokens']

In [ ]:
kwargs, total_tokens = answer_query("Give me three examples of blue t-shirts available on your catalogue.", simplified = True)

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['content'])

In [ ]:
total_tokens + result['total_tokens']

<a id='6-2'></a>

<a id='6-2'></a>
### 6.2 Logging Time!

In this step, you’ll implement a function to log user inputs, chatbot responses, and the total number of tokens used in the interaction.

You’ll use the **Pandas** library to create this log as a [**Pandas DataFrame**](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).

This function will act as a **logging step**, and should be called every time the user interacts with the chatbot.


In [ ]:
# Define the column names
columns = ['query', 'result', 'total_tokens', 'kwargs']

# Create an empty DataFrame with the specified columns
logging_dataset = pd.DataFrame(columns=columns)

In [ ]:
logging_dataset.head()

<a id='ex05'></a>

<a id='ex05'></a>
### Exercise 5
---

In this exercise, you’ll write a logging function that records information after each user interaction.

Your function will take the following inputs:

* `query`: the user’s original question
* `kwargs`: the dictionary returned by the `answer_question` function
* `total_tokens`: the total number of tokens used, which is the **sum** of:

  * the token count from `answer_question`
  * and the token count from the LLM’s response
* `result`: the LLM’s final answer, generated using `kwargs`
* `logging_dataset`: a `pandas.DataFrame` where the log entries will be saved
  (with columns: `'query'`, `'result'`, `'total_tokens'`, `'kwargs'`)

---
### Example:

```python
query = "What is your return policy?"
kwargs, total_tokens = answer_query(query)
result, result_tokens = generate_with_single_input(**kwargs)
# Log step:
generate_log(query, kwargs, total_tokens + result_tokens, result, logging_dataset)
```
---
<details>
    <summary style="color: green;"><strong>Hint 1</strong></summary>
Make sure to add the token count from the result to the one passed in.
</details>

<details>
    <summary style="color: green;"><strong>Hint 2</strong></summary>
To add a row to a DataFrame, you can create a dictionary like this:

```python
row = {
    "query": query,
    "result": result,
    "total_tokens": total_tokens,
    "kwargs": kwargs
}
```

Then use `logging_dataset.loc[len(logging_dataset)] = row` to append it.

</details>


In [ ]:
def generate_log(query, kwargs, total_tokens, result, logging_dataset):
    """
    Generates a log entry for a given query and its result, updating the total token count.

    Args:
        query (str): The search query or statement for which the log is being generated.
        kwargs (dict): A dictionary passed to the LLM
        total_tokens (int): The initial count of tokens before processing the result.
        result (dict): A dictionary containing the result details, including content and token count. result is obtained by calling the LLM with the kwargs
        logging_dataset (pandas.DataFrame): The dataset with columns 'query', 'result', 'total_tokens', 'kwargs' to store the results

    Returns:
        None: The function appends a new log entry to the 'logging_dataset' list but does not return any value.

    Notes:
        - This function updates the total token count by adding the total tokens from the result.
        - The log entry added to the 'logging_dataset' list consists of the query, answer, updated total tokens, and any additional keyword arguments.
    """
    ### START CODE HERE ###
    
    # Sum to the total tokens the tokens used in the llm call (it is under result['total_tokens'])
    total_tokens = None
    
    # Create a dictionary with the columns defined above, query, result, total_tokens and kwargs.
    
    row = None

    ### END CODE HERE ###

    
    # Append the row to the dataset
    logging_dataset.loc[len(logging_dataset)] = row

In [ ]:
# Test your code!

# simplified = True
query = "I want a casual look for hiking"
kwargs, total_tokens = answer_query(query, simplified = True)
result = generate_with_single_input(**kwargs)
generate_log(query, kwargs, total_tokens, result, logging_dataset)


# simplified = False
kwargs, total_tokens = answer_query(query, simplified = False)
result = generate_with_single_input(**kwargs)
generate_log(query, kwargs, total_tokens, result, logging_dataset)

In [ ]:
logging_dataset.head()

Expected output (results may vary)
```Python
                          query	                                           result            total_tokens	  kwargs
0	"What is your return policy?"	"{'role': 'assistant', 'content': 'Our return p..."  	        "553"	  "{'prompt': 'You will be provided with an FAQ f..."
```
                     

In [ ]:
unittests.test_generate_log(generate_log)

<a id='6-3'></a>

<a id='6-3'></a>
### 6.3 The ChatBot

Now you will run the ChatBot again! There are minor changes compared to the previous version to allow result logging. There are two versions available: one with the same behavior as in the previous assignment, and another with the new behavior.

We suggest trying the following queries to compare results and token usage:

* I bought a T-shirt and I didn't like it. Can I get a refund?
* I want a look to wear to a beach party at night. It's winter, and I'm a woman.

In [ ]:
chat_widget_standard = ChatWidget(generator_function = lambda x: answer_query(x, simplified = False), logging_function = generate_log)

Now, access the logging dataset:

In [ ]:
chat_widget_standard.chat_bot.logging_dataset

Now test the simplified version!

In [ ]:
chat_widget_simplified = ChatWidget(generator_function = lambda x: answer_query(x, simplified = True), logging_function = generate_log)

In [ ]:
chat_widget_simplified.chat_bot.logging_dataset

Congratulations! You improved your ChatBot using RAG techniques to reduce token count and added a logging system!